In [2]:
import pandas as pd
import numpy as np
import warnings
import joblib 
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, accuracy_score, roc_curve
from sklearn.impute import SimpleImputer
from imblearn.over_sampling import RandomOverSampler


df_clean = pd.read_csv(r'/workspaces/FinalYearProject/Matrimony_Matchmaker/Data-MGMT/completeresponse_dhs.csv')

# ==================================================
# 1. DEFINE TARGET COLUMNS & RED FLAGS
target_cols = [
    'Husband_partner_jealous_if_Wife_talks_with_other_men',
    'Husband_partner_insists_on_knowing_where_Wife_is',
    'Ever_been_humiliated_by_husband_partner',
    'Ever_been_insulted_or_made_to_feel_bad_by_husband_partner',
    'Ever_been_pushed,_shook_or_had_something_thrown_by_husband_partner',
    'Ever_been_slapped_by_husband_partner',
    "Person_who_usually_decides_on_Wife's_health_care",
    'Person_who_usually_decides_on_large_household_purchases',
    'Wife_afraid_of_husband_partner_most_of_the_time,_sometimes_or_never'
]

red_flag_answers = [
    'Yes', 'Often', 'Most of the time afraid', 'Husband/partner alone'
]


# Step B: Ordinal K-Means Clustering
severity_mapping = {
    'Never': 0, 'No': 0, 'Never afraid': 0, 'Respondent and husband/partner': 0, 'Respondent alone': 0,
    'Sometimes': 1, 'Yes, but not in the last 12 months': 1, 'Sometimes afraid': 1,
    'Often': 2, 'Yes': 2, 'Most of the time afraid': 2, 'Husband/partner alone': 2,
    "Don't know": -1, 'Other': -1, 'Someone else': -1, 'Unknown': -1
}

cat_imputer = SimpleImputer(strategy='constant', fill_value='Unknown')
df_targets_imputed = pd.DataFrame(cat_imputer.fit_transform(df_clean[target_cols]), columns=target_cols)

for col in target_cols:
    df_targets_imputed[col] = df_targets_imputed[col].map(severity_mapping).fillna(-1)

kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
raw_clusters = kmeans.fit_predict(df_targets_imputed)

# Ensure 1 is the high-risk cluster
df_targets_imputed['temp_cluster'] = raw_clusters
cluster_severity = df_targets_imputed.groupby('temp_cluster').mean().mean(axis=1)
high_risk_label = cluster_severity.idxmax()
df_clean['KMeans_Target'] = (raw_clusters == high_risk_label).astype(int)

# Step C: The Hybrid Target (If Rule OR K-Means flags it, it is High Risk)
df_clean['Marital_Stability_Target'] = df_clean['KMeans_Target']

print("Hybrid Target Distribution:")
print(df_clean['Marital_Stability_Target'].value_counts())

# ==================================================
# 2. PREPARE ML FEATURES
# ==================================================
X = df_clean.drop(columns=target_cols + ['Marital_Stability_Target', 'KMeans_Target'])
y = df_clean['Marital_Stability_Target']

numeric_cols = X.select_dtypes(include=['int64', 'float64']).columns
categorical_cols = X.select_dtypes(include=['object', 'category', 'str']).columns

num_imputer = SimpleImputer(strategy='median')
if len(numeric_cols) > 0: 
    X[numeric_cols] = num_imputer.fit_transform(X[numeric_cols])
if len(categorical_cols) > 0: 
    X[categorical_cols] = cat_imputer.fit_transform(X[categorical_cols])

X_encoded = pd.get_dummies(X, drop_first=True)
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.2, random_state=42, stratify=y)

# ==================================================
# 3. TRAIN WITH RANDOM OVERSAMPLING & OPTIMAL THRESHOLD
# ==================================================
ros = RandomOverSampler(random_state=42)
X_train_bal, y_train_bal = ros.fit_resample(X_train, y_train)

xgb = XGBClassifier(learning_rate=0.03, n_estimators=600, max_depth=6, eval_metric='logloss', random_state=42, n_jobs=-1)
xgb.fit(X_train_bal, y_train_bal)

# Find optimal threshold to boost both recalls
y_proba = xgb.predict_proba(X_test)[:, 1]
fpr, tpr, thresholds = roc_curve(y_test, y_proba)
optimal_idx = np.argmax(np.minimum(1 - fpr, tpr)) # Max-Min approach
optimal_threshold = thresholds[optimal_idx]

y_pred_optimal = (y_proba >= optimal_threshold).astype(int)

print("\n=== FINAL HYBRID MODEL PERFORMANCE ===")
print(f"Optimal Threshold Used: {optimal_threshold:.4f}")
print(f"Overall Accuracy: {accuracy_score(y_test, y_pred_optimal) * 100:.2f}%\n")
print(classification_report(y_test, y_pred_optimal, target_names=['Safe/Stable (0)', 'High Risk (1)']))


Hybrid Target Distribution:
Marital_Stability_Target
0    34905
1    11583
Name: count, dtype: int64

=== FINAL HYBRID MODEL PERFORMANCE ===
Optimal Threshold Used: 0.4927
Overall Accuracy: 60.10%

                 precision    recall  f1-score   support

Safe/Stable (0)       0.82      0.60      0.69      6981
  High Risk (1)       0.33      0.60      0.43      2317

       accuracy                           0.60      9298
      macro avg       0.58      0.60      0.56      9298
   weighted avg       0.70      0.60      0.63      9298


=== FINAL HYBRID MODEL PERFORMANCE ===
Optimal Threshold Used: 0.4927
Overall Accuracy: 60.10%

                 precision    recall  f1-score   support

Safe/Stable (0)       0.82      0.60      0.69      6981
  High Risk (1)       0.33      0.60      0.43      2317

       accuracy                           0.60      9298
      macro avg       0.58      0.60      0.56      9298
   weighted avg       0.70      0.60      0.63      9298



In [ ]:
import os
import pandas as pd

# ==================================================
# 4. CALCULATE COMBINED ORIGINAL FEATURE IMPORTANCE
# ==================================================
print("\n==================================================")
print("=== COMBINED ORIGINAL FEATURE IMPORTANCE RANKING ===")
print("==================================================")

# Extract the raw importance scores from the trained XGBoost model
raw_importances = xgb.feature_importances_
encoded_columns = X_train.columns

# Grab the original column names before they were split by get_dummies
original_columns = list(numeric_cols) + list(categorical_cols)

# Initialize a dictionary to hold the combined scores for the original features
aggregated_importance = {col: 0.0 for col in original_columns}

# Sort original columns by length descending to prevent substring matching bugs 
original_columns_sorted = sorted(original_columns, key=len, reverse=True)

# Map every split dummy column back to its original parent and add up the scores
for dummy_col, imp in zip(encoded_columns, raw_importances):
    for orig_col in original_columns_sorted:
        # Check if it's a numeric column (exact match) or a categorical dummy (starts with orig_col + '_')
        if dummy_col == orig_col or dummy_col.startswith(str(orig_col) + '_'):
            aggregated_importance[orig_col] += imp
            break

# Convert to a clean DataFrame and sort from highest to lowest
agg_importance_df = pd.DataFrame(list(aggregated_importance.items()), columns=['Original_Feature', 'Total_Importance'])
agg_importance_df = agg_importance_df.sort_values(by='Total_Importance', ascending=False).reset_index(drop=True)

# Exclude any feature that contains 'state' (case-insensitive)
mask_non_state = ~agg_importance_df['Original_Feature'].astype(str).str.lower().str.contains('state')
df_non_state = agg_importance_df[mask_non_state].copy().reset_index(drop=True)

# Calculate the new total importance sum EXCLUDING state features
total_non_state = df_non_state['Total_Importance'].sum()

if total_non_state <= 0:
    print('No non-state feature importances found or total importance is zero.')
else:
    # Compute percentage of total (out of the NON-STATE features)
    df_non_state['importance_pct'] = (df_non_state['Total_Importance'] / total_non_state * 100).round(3)

    # Provide a Top-N view and group the rest into 'Other' so the list sums to 100%
    top_n = 20
    top_df = df_non_state.head(top_n).copy()
    top_df = top_df[['Original_Feature', 'Total_Importance', 'importance_pct']]
    
    # Calculate what is left over for 'Other'
    rest_sum = df_non_state['Total_Importance'].sum() - top_df['Total_Importance'].sum()
    rest_pct = (rest_sum / total_non_state * 100).round(3)
    
    if rest_sum > 0:
        other_row = pd.DataFrame([{
            'Original_Feature': 'Other', 
            'Total_Importance': rest_sum, 
            'importance_pct': rest_pct
        }])
        top_plus = pd.concat([top_df, other_row], ignore_index=True)
    else:
        top_plus = top_df.copy()

    print(f"\nTop {top_n} original features (state excluded) with  aggregated:")
    print(top_plus.to_string(index=False))

    # Save CSV to resources for downstream use
    out_dir = '/workspaces/FinalYearProject/Matrimony_Matchmaker/resources/DHS dataset(48000)/xgboost'
    os.makedirs(out_dir, exist_ok=True)
    out_path = os.path.join(out_dir, 'combined_original_feature_importances_nonstate_pct.csv')

    try:
        top_plus.to_csv(out_path, index=False)
        print(f"\nSaved top+Other combined importances to:\n{out_path}")
    except Exception as e:
        print('\nFailed to save combined importances CSV:', e)


=== COMBINED ORIGINAL FEATURE IMPORTANCE RANKING ===

Top 20 original features (state excluded) with 'Other' aggregated:
            Original_Feature  Total_Importance  importance_pct
             Wife's Religion          0.070747       20.308001
            Husband Religion          0.053951       15.486000
 Wife's_occupation_(grouped)          0.051115       14.672000
Husband Occupation_(grouped)          0.050262       14.428000
   Husband's Education Level          0.027418        7.870000
      Wife's Education Level          0.025715        7.381000
           Husband Ethnicity          0.023063        6.620000
            Wife's Ethnicity          0.019355        5.556000
                   residence          0.007210        2.070000
          Wife's_current_age          0.006677        1.917000
  Wife's height(centimeters)          0.006618        1.900000
                 Husband_age          0.006246        1.793000

Saved top+Other combined importances to:
/workspaces/Final